In [10]:
import ee
import geemap
import geopandas as gpd
import pprint as pp
import pandas as pd

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')


roi_name = 'YKF_sub1'
resamp_method = 'bilinear'
resamp_res = 60
level = 'sr'


image_footprints_path = f'./data/overlap_dates_for_roi/{roi_name}_overlap_dates.shp'
best_image_dates = gpd.read_file(image_footprints_path) 
est_utm = f'EPSG:{best_image_dates.estimate_utm_crs().to_epsg()}'
roi_prefix = roi_name.split('_')[0]
region_shapes = gpd.read_file(f'./data/roi_shapes/rois/{roi_prefix}_sub_rois.shp')
full_roi_shape = region_shapes[region_shapes['sub_name'] == roi_name].iloc[0]


## Fetch the Image Collections

In [11]:
def convert_gpd_geom_to_ee(geom, est_utm):
    """
    Takes a geopandas geom object and coverts it to an Earth Engine polygon
    """
    if est_utm is None:
        out_crs = 'EPSG:4326'
    else:
        out_crs = est_utm

    coords = list(geom.exterior.coords)
    coords_list = [[x, y] for x, y in coords]
    return ee.Geometry.Polygon(coords_list, proj=out_crs)

def fetch_collections(
    polygon: ee.Geometry,
    date: str,
    date_plus1d: str,
    level: str
):
    
    def rescale_s2(img):
        rescaled_bands = img.divide(10_000)
        return rescaled_bands
    
    def rescale_ls8(img):
        rescaled_bands = img.multiply(0.0000275).add(-0.2)
        return rescaled_bands
    
    if level == 'sr':
        ls_asset = "LANDSAT/LC08/C02/T1_L2"
        s2_asset = "COPERNICUS/S2_SR_HARMONIZED"
        ls_bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']
    elif level == 'toa':
        ls_asset = "LANDSAT/LC08/C02/T1_TOA"
        s2_asset = "COPERNICUS/S2_HARMONIZED"
        ls_bands = ['B2', 'B3', 'B4', 'B5']
    else:
        print("Invalid level")

    s2_bands = ['B2', 'B3', 'B4', 'B8']

    ls8_col = ee.ImageCollection(ls_asset) \
        .filterDate(date, date_plus1d) \
        .filterBounds(polygon) \
        .select(ls_bands)

    ls8_col = ls8_col.map(rescale_ls8)
    ls8_col = ls8_col.map(lambda img: img.clip(polygon))


    # 2. Sentinel-2 ImageCollection mosaic
    s2_col = ee.ImageCollection(s2_asset) \
        .filterDate(date, date_plus1d) \
        .filterBounds(polygon) \
        .select(s2_bands) 

    s2_col = s2_col.map(rescale_s2)
    s2_col = s2_col.map(lambda img: img.clip(polygon))

    return s2_col, ls8_col


### Reproject the Collections

In [12]:
def repoject_collections(
        s2_col: ee.ImageCollection,
        ls8_col: ee.ImageCollection,
        est_utm: str, 
        resamp_method: str,
        resamp_res: int
    ):

    """
    Reprojects the Sentinel-2 and Landsat 8 image collections to the same UTM zone and resolution
    TODO: Worth adding workflow for the reduceResoltion() method ??
    """
    if resamp_res != 30:

        ls8_first_img = ls8_col.first()
        orig_ls8_proj = ls8_first_img.projection()
        #pp.pp(orig_ls8_proj.getInfo())
        ls8_utm_proj = orig_ls8_proj.atScale(resamp_res).getInfo()
        print("--- New Affine Transform @ 60m Res ---")
        #pp.pp(ls8_utm_proj)

        ls8_utm_col = ls8_col.map(lambda img: img.reproject(
            crs=est_utm,
            crsTransform=ls8_utm_proj['transform']
        ).resample(resamp_method))

    else:
        ls8_fist_img = ls8_col.first()
        ls_proj_orig = ls8_fist_img.projection().getInfo() 

        ls8_utm_col = ls8_col.map(lambda img: img.reproject(
            crs=est_utm,
            crsTransform=ls_proj_orig['transform'],
        ).resample(resamp_method))

        ls8_utm_proj = ls8_utm_col.first().projection().getInfo()

    # Sentinel-2 needs to be resampled to LandSat8's 60m or 30m grid
    s2_utm_col = s2_col.map(lambda img: img.reproject(
        crs=ls8_utm_proj['crs'],
        crsTransform=ls8_utm_proj['transform'],
    ).resample(resamp_method))


    return s2_utm_col, ls8_utm_col, ls8_utm_proj

## Make the Common Cloud Mask

### S2 Cloud Mask Mosiac

In [13]:
def make_s2_mask_mosaic(
    polygon: ee.Geometry,
    date: str,
    date_plus1d: str,
    
):
    """
    Generates a binary mask from image mosaics with the following mask criteria:
    1) Opaque clouds (probability > 70)
    2) Scene Classifacation Layer = cloud shaddows, cirrus, snow/ice
    Returns the binary mask and the % of each mask catagory
    """

    # Make a collection from the Sentinel-2 Cloud Probability Band
    s2_clouds = (ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
                 .filterBounds(polygon)
                 .filterDate(date, date_plus1d)
                 .select('probability')
                 .mosaic()
                 .clip(polygon)
    )
    
    # Make binary mask where probability > 70
    s2_clouds_binary = s2_clouds.gt(70).rename('s2_cloud_mask')
    # Calculate the Image's Cloud Fraction
    cloud_stats = s2_clouds_binary.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=10
    ).getInfo()
    cloud_faction = cloud_stats.get('s2_cloud_mask', -1)

    # Make collction from the Sentinel-2 SCL (Scene Classifacation Layer) Band
    s2_scl = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(polygon)
              .filterDate(date, date_plus1d)
              .select('SCL')
              .mosaic()
              .clip(polygon)
    )
    # Make binary masks for shaddows, cirrus, and snow/ice
    s2_shaddows_mask = s2_scl.eq(3).rename('s2_shaddow_mask')
    s2_cirrus_mask = s2_scl.eq(10).rename('s2_cirrus_mask')
    s2_snowice_mask = s2_scl.eq(11).rename('s2_snowice_mask')
    # Get fractions for shaddows, cirrus, and snow/ice
    shaddow_stats = s2_shaddows_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=10
    ).getInfo()
    shaddow_fraction = shaddow_stats.get('s2_shaddow_mask', -1)
    cirrus_stats = s2_cirrus_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=10
    ).getInfo()
    cirrus_fraction = cirrus_stats.get('s2_cirrus_mask', -1)
    snowice_stats = s2_snowice_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=10
    ).getInfo()
    snowice_fraction = snowice_stats.get('s2_snowice_mask', -1)

    # Make a full mask where any of the above masks are true
    s2_full_mask = s2_clouds_binary.Or(s2_shaddows_mask).Or(s2_cirrus_mask).Or(s2_snowice_mask).rename('s2_full_mask')
    # Get the fraction of the image that is masked
    full_stats = s2_full_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=10
    ).getInfo()
    full_mask_fraction = full_stats.get('s2_full_mask', -1)

    # Make a dictionary to hold the mosaic's mask stats
    s2_mask_stats = {
        's2_cloud_fraction': cloud_faction,
        's2_shaddow_fraction': shaddow_fraction,
        's2_cirrus_fraction': cirrus_fraction,
        's2_snowice_fraction': snowice_fraction,
        's2_full_mask_fraction': full_mask_fraction
    }

    return s2_full_mask, s2_mask_stats


### LS8 Cloud Mask Mosaic

In [14]:
def make_ls8_mask_mosaic(
        polygon: ee.Geometry,
        date: str,
        date_plus1d: str,
):
    """
    Generates a binary mask from image mosaics with the following mask criteria:
    QA_PIXEL = Clouds, Cloud Shadows, Snow/Ice, Cirrus
    Returns the binary mask and the % of each mask catagory
    """
    # Make a collection from the Landsat 8 QA Band
    # Worth noting this band is the same for both SR and TOA data collections
    ls8_qa = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
              .filterBounds(polygon)
              .filterDate(date, date_plus1d)
              .select('QA_PIXEL')
              .mosaic()
              .clip(polygon)
    )
    # Define the cloud, cloud shadow, snow/ice, and cirrus bitmasks
    cloud_bit_mask = 1 << 3
    shaddow_bit_mask = 1 << 4
    snowice_bit_mask = 1 << 5
    cirrus_bit_mask = 1 << 2
    full_bitmask = (cloud_bit_mask | shaddow_bit_mask | snowice_bit_mask | cirrus_bit_mask)

    # Make binary masks for clouds, cloud shaddows, snow/ice, and cirrus
    # Get fractions for each mask
    ls8_cloud_mask = ls8_qa.bitwiseAnd(cloud_bit_mask).neq(0).rename('ls8_cloud_mask')
    cloud_stats = ls8_cloud_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=30
    ).getInfo()
    cloud_fraction = cloud_stats.get('ls8_cloud_mask', -1)
    # Shaddows 
    ls8_shaddow_mask = ls8_qa.bitwiseAnd(shaddow_bit_mask).neq(0).rename('ls8_shaddow_mask')
    shaddow_stats = ls8_shaddow_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=30
    ).getInfo()
    shaddow_fraction = shaddow_stats.get('ls8_shaddow_mask', -1)
    # Snow/Ice
    ls8_snowice_mask = ls8_qa.bitwiseAnd(snowice_bit_mask).neq(0).rename('ls8_snowice_mask')
    snowice_stats = ls8_snowice_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=30
    ).getInfo()
    snowice_fraction = snowice_stats.get('ls8_snowice_mask', -1)
    # Cirrus
    ls8_cirrus_mask = ls8_qa.bitwiseAnd(cirrus_bit_mask).neq(0).rename('ls8_cirrus_mask')
    cirrus_stats = ls8_cirrus_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=30
    ).getInfo()
    cirrus_fraction = cirrus_stats.get('ls8_cirrus_mask', -1)
    # The full mask
    ls8_full_mask = ls8_qa.bitwiseAnd(full_bitmask).neq(0).rename('ls8_full_mask')
    full_stats = ls8_full_mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=30
    ).getInfo()
    full_mask_fraction = full_stats.get('ls8_full_mask', -1)

    ls8_mask_stats = {
        'ls8_cloud_fraction': cloud_fraction,
        'ls8_shaddow_fraction': shaddow_fraction,
        'ls8_snowice_fraction': snowice_fraction,
        'ls8_cirrus_fraction': cirrus_fraction,
        'ls8_full_mask_fraction': full_mask_fraction
    }

    return ls8_full_mask, ls8_mask_stats

### Combine S2 and LS8 Cloud Masks

In [15]:
def reduce_mask_resolution(
        mask: ee.Image, # Technically a mosaic
        projection_utm: dict, # Need the detialed affine transformation
):
    """
    Reduces the resolution of a mask mosaic to the desired resolution
    with nearest neighbor resampling
    """

    mask_reproj = mask.reproject(
        crs=projection_utm['crs'],
        crsTransform=projection_utm['transform'],
    )
    # Default is nearest neighbor without specifying resample method
    return mask_reproj

def calc_common_mask_frac(
    common_mask: ee.Image,
    polygon: ee.Geometry,
    resamp_res: int
):
    """
    Need to reproject the common mask from UTM to EPSG:4326 to find masked fraction in image footprint
    Converting the image footprint to UTM causes some geometry errors in Earth Engine
    """
    common_mask_reproj = common_mask.reproject(
        crs="EPSG:4326",
        scale=resamp_res
    )
    # Get the number of pixels for the dilated mask
    common_mask_stats = common_mask_reproj.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=polygon,
        maxPixels=1e13,
        scale=resamp_res
    ).getInfo()

    common_mask_fraction = common_mask_stats.get('combined_mask', -1)

    return common_mask_fraction


def generate_common_mask(
    polygon: ee.Geometry,
    date: str,
    date_plus1d: str,
    resamp_res: int,
    projection_utm: dict
):
    """
    Generates a common mask for the two satellite's image mosaics
    1) Sieve the mask to remove small isolated pixels (less than 50 pixels)
    2) Dialate the mask to be more conservative (500 meters)
    """

    s2_mask, s2_stats = make_s2_mask_mosaic(polygon, date, date_plus1d)
    ls8_mask, ls8_stats = make_ls8_mask_mosaic(polygon, date, date_plus1d)
    
    # Reproject the masks to the same UTM zone and resolution
    s2_mask = reduce_mask_resolution(s2_mask, projection_utm)
    ls8_mask= reduce_mask_resolution(ls8_mask, projection_utm)

    # Make a common mask for the two Satellite's
    s2_mask = s2_mask.rename('combined_mask')
    ls8_mask = ls8_mask.rename('combined_mask')
    combined_mask = s2_mask.Or(ls8_mask)

    # Sieve and dialate the common mask
    connected_pixels = combined_mask.connectedPixelCount(maxSize=1_000, eightConnected=True)
    sieved_mask = combined_mask.updateMask(connected_pixels.gte(50))
    dilation_kernal = ee.Kernel.circle(radius=1_000, units='meters', normalize=False)
    dilated_mask = sieved_mask.focal_max(kernel=dilation_kernal, iterations=1)

    # Get the combined, S2 and LS8 dictionary of mask stats
    mask_stats = s2_stats | ls8_stats
    common_mask_fraction = calc_common_mask_frac(dilated_mask, polygon, resamp_res)
    mask_stats['combined_mask_fraction'] = common_mask_fraction
    mask_stats['date'] = date
    mask_stats['resamp_res'] = resamp_res

    return dilated_mask, mask_stats
    


## Apply the cloud mask to the exported images

## Export the masked Images

In [16]:
def export_img_collections(
    s2_col: ee.ImageCollection,
    ls8_col: ee.ImageCollection,
    roi_box: ee.Geometry,
    projection_info: dict,
    roi_name: str,
    date: str,
    resamp_method: str,
    resamp_res: int,
    level: str
):
    """ 
    Exports the Sentinel-2 and Landsat 8 image collections to Google Drive
    """

    if level == 'sr':
        folder = 'sr_images'
    elif level == 'toa':
        folder = 'toa_images'
    else:
        print("Invalid level")


    s2_size = s2_col.size().getInfo()
    ls8_size = ls8_col.size().getInfo()

    for i in range(s2_size):
        # Make export_name
        out_img = ee.Image(s2_col.toList(s2_size).get(i))
        export_name = f'Sentinel2_{level}_date_{date}_roi_{roi_name}_resampled_{resamp_method}{resamp_res}_idx{i}'
        task = ee.batch.Export.image.toDrive(
            image=out_img,
            description=export_name,
            fileNamePrefix=export_name,
            folder=folder,
            region=roi_box,
            crs=projection_info['crs'],
            crsTransform=projection_info['transform']
        )
        task.start()
        print(export_name)

    for i in range(ls8_size):
        out_img = ee.Image(ls8_col.toList(ls8_size).get(i))
        export_name = f'Landsat8_{level}_date_{date}_roi_{roi_name}_resampled_{resamp_method}{resamp_res}_idx{i}'
        task = ee.batch.Export.image.toDrive(
            image=out_img,
            description=export_name,
            fileNamePrefix=export_name,
            folder=folder,
            region=roi_box,
            crs=projection_info['crs'],
            crsTransform=projection_info['transform']
        )
        task.start()
        print(export_name)
    


### Full Processing Function

In [ ]:
def pair_processor(
    est_utm: str,
    footprint: gpd.GeoSeries,
    full_roi: gpd.GeoSeries,
    resamp_method: str,
    resamp_res: int,
    level: str
):

    date = footprint.date
    geom = footprint.geometry 
    roi_name = full_roi.sub_name
    date_plus1d = (pd.to_datetime(date) + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    polygon = convert_gpd_geom_to_ee(geom, None)
    s2_col, ls8_col = fetch_collections(polygon, date, date_plus1d, level)

    s2_utm_col, ls8_utm_col, ls8_utm_proj = repoject_collections(
        s2_col=s2_col, 
        ls8_col=ls8_col, 
        est_utm=est_utm, 
        resamp_method=resamp_method,
        resamp_res=resamp_res,
    )

    common_mask, mask_stats = generate_common_mask(
        polygon=polygon,
        date=date,
        date_plus1d=date_plus1d,
        resamp_res=resamp_res,
        projection_utm=ls8_utm_proj,
    )

    masked_percent = mask_stats['combined_mask_fraction'] * 100
    mask_stats['level'] = level
    mask_stats['resamp_method'] = resamp_method
    mask_stats['roi_name'] = roi_name
    if masked_percent > 70:
        print(f"Bad Image Set: {masked_percent:.2f}% masked")
        return mask_stats

    print(f"Exporting Good Image -- Masked % {masked_percent:.2f}")
    s2_utm_col_masked = s2_utm_col.map(lambda img: img.updateMask(common_mask.eq(0)))
    ls8_utm_col_masked = ls8_utm_col.map(lambda img: img.updateMask(common_mask.eq(0)))

    full_roi_poly = convert_gpd_geom_to_ee(full_roi.geometry, None)
    bounds_poly = full_roi_poly.bounds()
    bounds_poly_utm = bounds_poly.transform(ee.Projection(est_utm), 1)

    export_img_collections(
        s2_col=s2_utm_col_masked, 
        ls8_col=ls8_utm_col_masked, 
        roi_box=bounds_poly_utm, 
        projection_info=ls8_utm_proj, 
        roi_name=roi_name,
        date=date,
        resamp_method=resamp_method, 
        resamp_res=resamp_res,
        level=level
    )
    print("-----------------------------------------------------")

    return mask_stats


In [18]:
all_mask_stats = []
for idx, row in best_image_dates.iterrows():
    footprint = row
    mask_stats = pair_processor(
        est_utm=est_utm, 
        footprint=footprint, 
        full_roi=full_roi_shape, 
        resamp_method=resamp_method, 
        resamp_res=resamp_res,
        level=level
    )
    print(f'processing {idx + 1} of {len(best_image_dates)}')
    all_mask_stats.append(mask_stats)
    

--- New Affine Transform @ 60m Res ---
Exporting Good Image -- Masked % 21.35
Sentinel2_sr_date_2018-06-07_roi_YKF_sub1_resampled_bilinear60_idx0
Sentinel2_sr_date_2018-06-07_roi_YKF_sub1_resampled_bilinear60_idx1
Landsat8_sr_date_2018-06-07_roi_YKF_sub1_resampled_bilinear60_idx0
-----------------------------------------------------
processing 1 of 28
--- New Affine Transform @ 60m Res ---
Bad Image Set: 99.97% masked
processing 2 of 28
--- New Affine Transform @ 60m Res ---
Exporting Good Image -- Masked % 4.27
Sentinel2_sr_date_2019-05-16_roi_YKF_sub1_resampled_bilinear60_idx0
Sentinel2_sr_date_2019-05-16_roi_YKF_sub1_resampled_bilinear60_idx1
Landsat8_sr_date_2019-05-16_roi_YKF_sub1_resampled_bilinear60_idx0
Landsat8_sr_date_2019-05-16_roi_YKF_sub1_resampled_bilinear60_idx1
-----------------------------------------------------
processing 3 of 28
--- New Affine Transform @ 60m Res ---
Exporting Good Image -- Masked % 45.05
Sentinel2_sr_date_2019-05-23_roi_YKF_sub1_resampled_bilinear6

In [ ]:
mask_data = pd.DataFrame(all_mask_stats)
mask_data.to_csv(f'./data/img_mask_solar_stats/{roi_name}_{level}_resampled_{resamp_method}{resamp_res}.csv', index=False)

In [25]:
mask_data.head(25)

,s2_cloud_fraction,s2_shaddow_fraction,s2_cirrus_fraction,s2_snowice_fraction,s2_full_mask_fraction,ls8_cloud_fraction,ls8_shaddow_fraction,ls8_snowice_fraction,ls8_cirrus_fraction,ls8_full_mask_fraction,combined_mask_fraction,date,resamp_res,level,roi_name,resamp_method
0,0.000000e+00,0.000000,0.076274,0.000000,0.076274,0.001003,0.000344,0.000000,0.001662,0.002449,0.213537,2018-06-07,60,sr,YKF_sub1,bilinear
1,3.647331e-01,0.047454,0.036232,0.378919,0.768084,0.547052,0.187983,0.170955,0.234489,0.905991,0.999742,2019-05-09,60,sr,YKF_sub1,bilinear
2,2.394807e-07,0.000415,0.000235,0.000466,0.001116,0.000552,0.002696,0.002634,0.000000,0.005883,0.042682,2019-05-16,60,sr,YKF_sub1,bilinear
3,1.124090e-01,0.062019,0.050482,0.000035,0.207283,0.080211,0.087469,0.000000,0.040791,0.168258,0.450525,2019-05-23,60,sr,YKF_sub1,bilinear
4,1.101458e-01,0.108170,0.008032,0.001523,0.222827,0.059092,0.112298,0.000087,0.013556,0.171586,0.633283,2019-06-08,60,sr,YKF_sub1,bilinear
5,1.108200e-02,0.005145,0.011320,0.000000,0.027180,0.010533,0.013948,0.000000,0.000373,0.024492,0.180489,2019-06-17,60,sr,YKF_sub1,bilinear
6,8.834542e-02,0.013713,0.084999,0.000000,0.140263,0.117781,0.071078,0.000057,0.081983,0.192884,0.452755,2019-08-11,60,sr,YKF_sub1,bilinear
7,1.534644e-01,0.155001,0.006253,0.000119,0.312867,0.124220,0.139714,0.000019,0.000000,0.263953,0.776223,2019-09-05,60,sr,YKF_sub1,bilinear
8,3.250095e-06,0.000002,0.001740,0.960608,0.962352,0.001609,0.002047,0.991375,0.000000,0.995032,1.000000,2020-05-02,60,sr,YKF_sub1,bilinear
9,0.000000e+00,0.000000,0.000045,0.000003,0.000048,0.000025,0.000000,0.000010,0.000000,0.000035,0.000000,2020-05-18,60,sr,YKF_sub1,bilinear
